# Named Entityy Recogninatoion

In [1]:
import spacy
from nltk import sent_tokenize


In [2]:
!python -m spacy download en_core_web_trf

     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.3/457.4 MB ? eta -:--:--
     ---------------------------------------- 1.0/457.4 MB 2.8 MB/s eta 0:02:44
     ---------------------------------------- 1.8/457.4 MB 3.2 MB/s eta 0:02:21
     ---------------------------------------- 2.6/457.4 MB 3.5 MB/s eta 0:02:10
     ---------------------------------------- 3.4/457.4 MB 3.7 MB/s eta 0:02:02
     ---------------------------------------- 4.5/457.4 MB 3.7 MB/s eta 0:02:02
     ---------------------------------------- 5.0/457.4 MB 3.5 MB/s eta 0:02:09
     ---------------------------------------- 5.5/457.4 MB 3.4 MB/s eta 0:02:12
      --------------------------------------- 6.3/457.4 MB 3.4 MB/s eta 0:02:13
      --------------------------------------- 6.8/457.4 MB 3.3 MB/s eta 0:02:17
      --------------------------------------- 7.6/457.4 MB 3.4 MB/s eta 0:02:13
      --------------------------------------- 8.4/457.


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Load Model

In [3]:
def load_model():
    nlp = spacy.load("en_core_web_trf")
    return nlp

In [4]:
nlp_model = load_model()

d:\NLP Analyzer\analyzer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Dataset

In [5]:
import os
import sys
import pathlib
folder_path = pathlib.Path().parent.resolve()
sys.path.append(os.path.join(folder_path, '../'))
from utils import load_subtitles_dataset

In [6]:
dataset_path = "../data/Subtitles/"
df = load_subtitles_dataset(dataset_path)

In [7]:
df.head()

,episode,script
0,1,"A long time ago, a powerful demon fox appeared..."
1,2,"C'mon!\n Running like a fugitive,\n Being chas..."
2,3,"C'mon!\n Running like a fugitive,\n Being chas..."
3,4,"C'mon!\n Running like a fugitive,\n Being chas..."
4,5,"C'mon!\n Running like a fugitive,\n Being chas..."


In [8]:
sample_script = df.iloc[1]['script']
sample_script

'C\'mon!\n Running like a fugitive,\n Being chased by something\n Inside my heart is pounding\n My throat dry like it\'s withering\n For no single one,\n To none does belong,\n This time is ours, right now...\n Unraveling the pain,\n Unraveling our hearts,\n Unraveling shadows\n Stifling our breath,\n Reaching for beyond,\n Tearing through the dark\n In fighting and in love\n To the distant light above,\n Yes, we are on the way\n I wanna rock...\n Rockin\' my heart\n Hey, you!\n Are you sure you want to look like that?\n Yes, of course! C\'mon! So hurry hurry!\n Don\'t regret it later.\n Say cheese!\n My Name Is Konohamaru!\n I couldn\'t decide on how to look.\n It took me 3 hours to decide.\n But as you can see, it\'s more like a work of art.\n Doesn\'t it look great?\n Take it over.\n What?!\n Take it over.\n Stop saying that!\n Transform!\n Oh please, Lord Hokage…\n The Sexy Jutsu… That\'s quite a devious technique.\n By the way Naruto, where\'s your headband?\n I won\'t wear it unt

In [9]:
sentences = sent_tokenize(sample_script)

In [10]:
sentences = sentences[60:90]

In [11]:
sentence = ".".join(sentences)

In [12]:
sentence

'Then allow me to teach you Ninjutsu,\n so that your dream will become a reality..That\'s right..Stick with me and you will gain a shortcut towards becoming the Fifth Hokage..Is that clear, Honorable Grandson?.He\'s gone!.It seems he\'s gone after Naruto..Oh no!.This is terrible!.Honorable Grandson!.How did he become like this?.That was the 20th sneak attack today..And now he\'s in the company of Naruto..I\'m a bit worried..I just hope he doesn\'t pick up more foolishness..Huh?.Stop following me!.What is it now?!.Your camouflage is pathetic..You\'ve seen through my disguise....Your reputation is well-earned..I will allow you to make me your apprentice..But you must first teach me that  "Sexy Jutsu"  technique\n you used to beat Grandpa Hokage..You gotta be kidding..I beg you to say yes, boss!.Huh?.Boss?.Boss!.Boss!.Boss!.I guess I have no choice.'

# Run Model

In [13]:
doc = nlp_model(sentence)


In [14]:
doc.ents

(Fifth, Naruto, 20th, today, Naruto, Hokage)

In [15]:
for entity in doc.ents:
    print(entity, entity.label_)

Fifth ORDINAL
Naruto PERSON
20th ORDINAL
today DATE
Naruto PERSON
Hokage PERSON


In [59]:
def get_ners_inference(script):
    script_sentences = sent_tokenize(script)

    ner_output = []

    for sentence in script_sentences:
        doc = nlp_model(sentence)
        ners = set()
        for entity in doc.ents:
            if entity.label_ == "PERSON":
                # Clean and extract the name
                first_name = entity.text.strip().split()[0]

                # ADD IT TO THE SET (Choose first_name or entity.text)
                ners.add(first_name)

        # Convert set to list before saving if needed by downstream code
        ner_output.append(list(ners))

    return ner_output

In [60]:
df = df.head(10)
df

,episode,script
0,1,"A long time ago, a powerful demon fox appeared..."
1,2,"C'mon!\n Running like a fugitive,\n Being chas..."
2,3,"C'mon!\n Running like a fugitive,\n Being chas..."
3,4,"C'mon!\n Running like a fugitive,\n Being chas..."
4,5,"C'mon!\n Running like a fugitive,\n Being chas..."
5,6,"C'mon!\n Running like a fugitive,\n Being chas..."
6,7,"C'mon!\n Running like a fugitive,\n Being chas..."
7,8,"C'mon!\n Running like a fugitive,\n Being chas..."
8,9,"C'mon!\n Running like a fugitive,\n Being chas..."
9,12,"C'mon!\n Running like a fugitive,\n Being chas..."


In [61]:
df['ners'] = df['script'].apply(get_ners_inference)

In [62]:
df

,episode,script,ners
0,1,"A long time ago, a powerful demon fox appeared...","[[], [], [], [], [], [], [], [Naruto], [], [],..."
1,2,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [], [], [], [], [], [], [Konohama..."
2,3,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [Sasuke, Sakura], [], [Konohamaru..."
3,4,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [Naruto], [], [], [Iruka], [], [N..."
4,5,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [], [], [], [], [], [], [], [], [..."
5,6,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [Sasuke], [], [Naruto], [], [Naruto],..."
6,7,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [], [], [], [], [], [], [], [], [..."
7,8,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [], [], [], [], [Sasuke], [], [],..."
8,9,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [], [], [], [], [], [], [], [], [..."
9,12,"C'mon!\n Running like a fugitive,\n Being chas...","[[], [], [], [], [Zabuza], [], [], [], [Naruto..."


# Character Network

In [63]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx 
from pyvis.network import Network

In [64]:
def generate_character_network(df):

    windows=10
    entity_relationship = []

    for row in df['ners']:
        previous_entities_in_window = []

        for sentence in row:
            previous_entities_in_window.append(list(sentence))
            previous_entities_in_window = previous_entities_in_window[-windows:]

            # Flatten 2D List into 1D List
            previous_entities_flattened = sum(previous_entities_in_window, [])

            for entity in sentence:
                for entity_in_window in previous_entities_flattened:
                    if entity != entity_in_window:
                        entity_relationship.append(sorted([entity, entity_in_window]))
    
    relationship_df = pd.DataFrame({'value': entity_relationship})
    relationship_df['source'] = relationship_df['value'].apply(lambda x: x[0])
    relationship_df['target'] = relationship_df['value'].apply(lambda x: x[1])
    relationship_df = relationship_df.groupby(['source', 'target']).count().reset_index()
    relationship_df = relationship_df.sort_values('value', ascending=False)

    return relationship_df

In [65]:
relationship_df = generate_character_network(df)

In [66]:
relationship_df

,source,target,value
164,Naruto,Sasuke,122
203,Sakura,Sasuke,69
90,Iruka,Naruto,45
163,Naruto,Sakura,40
155,Mizuki,Naruto,29
...,...,...,...
179,Narutos,jonin,1
86,Ino,Sasuke,1
38,Genin,Naruto,1
39,Genin,Ninja,1


In [67]:
relationship_df = relationship_df.sort_values('value', ascending=False)
relationship_df = relationship_df.head(200)

In [68]:
G = nx.from_pandas_edgelist(
    relationship_df,
    source='source',
    target='target',
    edge_attr='value',
    create_using=nx.Graph()
)

net = Network(notebook=True, width="1000px", height="700px", bgcolor="#222222", font_color="white", cdn_resources="remote")
node_degree = dict(G.degree)
nx.set_node_attributes(G, node_degree, 'size')
net.from_nx(G)
net.show("naruto.html")

naruto.html
